# Lab 12: Statistical Testing with Random Graph Null Models

In earlier labs, we compared observed networks to random graphs informally.
In this lab, that comparison becomes explicit statistical testing.

We will:
- choose a graph statistic,
- state a null hypothesis $\mathcal{H}_0$,
- state an alternative hypothesis $\mathcal{H}_1$,
- generate many graphs from a null model,
- compute the statistic under the null model,
- compare the observed graph to that null distribution,
- estimate an empirical $p$-value.

Our goal is not only to say that an observed graph “looks unusual,” but to ask whether it appears unusual relative to a clearly stated null model.

In [ ]:
from scratch.old_labs.module_12.lab12_helpers import *

### Part 1: The observed graph and the null model

We begin with an observed graph: Zachary's Karate Club graph.

Our null model will be $G(n,m)$: a random graph with the same number of vertices and edges as the observed graph.

This means the null model preserves graph size, but places the edges at random.

In [ ]:
K = karate_graph()
observed_metrics = graph_metrics(K)
observed_metrics

For the clustering test, we will use:


$\mathcal{H}_0$: The observed graph is consistent with a $G(n,m)$ random graph model.


$\mathcal{H}_1$: The observed graph has larger average clustering than expected under $G(n,m)$.

This is a one-sided test because we are specifically asking whether the observed clustering is unusually high.

### Part 2: Simulating the null distribution

To test the hypothesis, we generate many random graphs from the null model and compute the same statistic on each one. Here, we will simulate 500 times. This is likely overkill for such a simple example, but imagine using this workflow on a large, complex graph system. Minor deviations in the structure across random trials can give untrustworthy statistics, so we hope to "smooth those out" by oversampling.

This gives a null distribution for average clustering.

In [ ]:
null_df = simulate_matching_observed(K, trials=500, seed=12)
null_df.head()

In [ ]:
observed_clustering = observed_metrics["average_clustering"]
observed_path_length = observed_metrics["average_path_length"]

print(f"Observed clustering: {observed_clustering}, \nObserved path length: {observed_path_length}")
print("\n ****** Null statistics ******")
null_df["average_clustering"].describe()

### Part 3: Visualizing the null distribution

Now compare the observed clustering coefficient to the distribution of clustering coefficients under the null model.

In [ ]:
plot_metric_histogram(
    null_df["average_clustering"],
    observed_value=observed_clustering,
    xlabel="Average clustering",
    title="Null distribution of average clustering"
)

Very clearly the observed value is way outside the null distribution, but we will continue setting up the workflow for application to harder cases later.

### Part 4: Estimating an empirical $p$-value

For this one-sided test, the empirical $p$-value is the proportion of null-model graphs whose clustering coefficient is at least as large as the observed one.

A very small $p$-value means that the observed clustering would be rare under the null model.

In [ ]:
empirical_p_value_upper(null_df["average_clustering"], observed_clustering)

Interpretation questions:

- If the empirical $p$-value is very small, what does that say about $\mathcal{H}_0$?
- Does it prove that the null model is false?
- Or does it say that the observed graph would be unusual if $\mathcal{H}_0$ were true?

### Part 5: A second test using average path length

Now repeat the process using average path length.

This time, we may not know in advance whether the observed graph should be larger or smaller than the null-model average path length, so a two-sided interpretation is more natural.

In [ ]:
plot_metric_histogram(
    null_df["average_path_length"],
    observed_value=observed_path_length,
    xlabel="Average path length",
    title="Null distribution of average path length"
)

In [ ]:
empirical_p_value_two_sided(null_df["average_path_length"], observed_path_length)

For this test, use:

$\mathcal{H}_0$: The observed graph is consistent with the $G(n,m)$ null model in average path length.

$\mathcal{H}_1$: The observed graph has an average path length that is unusually different from the null model.

Questions:
- Does the observed path length look unusual?
- Is clustering or path length the stronger signal against the null model?
- Why might one statistic show stronger evidence than another?

### Part 6: The null model matters

A hypothesis test depends on the null model.

Now compare the same observed graph to a different null model, $G(n,p)$, where $p$ is chosen so that the expected number of edges matches the observed graph.

In [ ]:
n = K.number_of_nodes()
m = K.number_of_edges()
# let's define an "inflation factor" that forces more connections
# play around with some larger and smaller factors to see the influence on the p-value
inf_factor = 8.0
p = (inf_factor * m) / (n * (n - 1))
p

In [ ]:
gnp_df = simulate_gnp_metrics(n, p, trials=500, seed=21)
gnp_df.head()

In [ ]:
plot_metric_histogram(
    gnp_df["average_clustering"],
    observed_value=observed_clustering,
    xlabel="Average clustering",
    title="Null distribution under G(n,p)"
)

In [ ]:
empirical_p_value_upper(gnp_df["average_clustering"], observed_clustering)

Questions:
- Does the $p$-value change when the null model changes?
- Why is hypothesis testing always tied to modeling assumptions?
- What does this tell you about statistical evidence in network science?